In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

## Features

In [4]:
root = 'BrainLat/dataset/EEG data/'

In [5]:
"""common_channels = []
for disease_label in os.listdir(root):
    # Skip non-folder files
    if "BrainLat" in disease_label or "SYNAPSE" in disease_label:
        continue
    print(disease_label)
    disease_label_path = os.path.join(root, disease_label)
    for file_folder in os.listdir(disease_label_path):
        if file_folder == 'AR' or file_folder == 'CL':
            print(file_folder)
            file_folder_path = os.path.join(disease_label_path, file_folder)
            for file in os.listdir(file_folder_path):
                file_path = os.path.join(file_folder_path, file)
                set_file_path = ''
                if 'sub' in file_path or 'suj' in file_path:   # 4_MS has different folder name
                    set_file_path = os.path.join(file_path, 'eeg')
                    if not os.path.exists(set_file_path):  # same subjects in 4_MS has different folder structure
                        set_file_path = file_path
                    # print(set_file_path)
                    for set_file in os.listdir(set_file_path):
                        if set_file.endswith('.set'):
                            set_file_path = os.path.join(set_file_path, set_file)
                            print(set_file_path)
                            try:
                                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
                                current_channels = set(raw.info['ch_names'])
                                if not common_channels:
                                    common_channels = current_channels
                                else:
                                    common_channels &= current_channels

                                print('\n')
                            except:
                                print('Load EEG data error, skip this subject\n')
                                continue
                        else:  # .ftd file will be read if .set file call it. no need to read it specifically
                            continue
    print("-------------------------------------\n")
common_channels = list(common_channels)
print(common_channels)
print("Common channels number: ", len(common_channels))"""

'common_channels = []\nfor disease_label in os.listdir(root):\n    # Skip non-folder files\n    if "BrainLat" in disease_label or "SYNAPSE" in disease_label:\n        continue\n    print(disease_label)\n    disease_label_path = os.path.join(root, disease_label)\n    for file_folder in os.listdir(disease_label_path):\n        if file_folder == \'AR\' or file_folder == \'CL\':\n            print(file_folder)\n            file_folder_path = os.path.join(disease_label_path, file_folder)\n            for file in os.listdir(file_folder_path):\n                file_path = os.path.join(file_folder_path, file)\n                set_file_path = \'\'\n                if \'sub\' in file_path or \'suj\' in file_path:   # 4_MS has different folder name\n                    set_file_path = os.path.join(file_path, \'eeg\')\n                    if not os.path.exists(set_file_path):  # same subjects in 4_MS has different folder structure\n                        set_file_path = file_path\n               

In [6]:
common_channels = ['A28', 'C20', 'D4', 'C22', 'D16', 'A21', 'A26', 'A30', 'A31', 'C6', 'C27', 'D1', 'C12', 'D15', 'C9', 'A5', 'D2', 'B15', 'B17', 'A7', 'D20', 'D24', 'C7', 'A2', 'B12', 'B30', 'B11', 'D11', 'B23', 'D19', 'A20', 'A1', 'B21', 'C2', 'A17', 'A13', 'C11', 'D31', 'C19', 'C21', 'C32', 'D13', 'B3', 'C25', 'D8', 'B25', 'A12', 'C31', 'B22', 'B27', 'C5', 'D10', 'C17', 'A14', 'C16', 'D21', 'D22', 'A32', 'C23', 'D28', 'D14', 'B16', 'B1', 'D23', 'B5', 'D3', 'B24', 'D12', 'D17', 'A18', 'C8', 'A27', 'A24', 'B6', 'B18', 'A29', 'A6', 'D6', 'C10', 'B9', 'C18', 'A11', 'A3', 'C14', 'D9', 'D30', 'D25', 'B19', 'A9', 'B20', 'B26', 'D5', 'B8', 'C3', 'D27', 'A16', 'C26', 'C4', 'D26', 'B7', 'B29', 'A19', 'A8', 'A4', 'B2', 'B10', 'A25', 'B32', 'C28', 'C29', 'C30', 'D7', 'D32', 'A22', 'C13', 'B13', 'D18', 'A10', 'A15', 'A23', 'B31', 'C24', 'C1', 'B14', 'C15', 'D29', 'B28', 'B4']

In [7]:
def data_preprocessing(
    raw: mne.io.Raw,
    ch_names: list,
    montage: str,
    notch_freq: float = 50.0,
    l_freq: float = 0.5,
    h_freq: float = 45.0,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        raw (mne.io.Raw): MNE Raw object containing EEG data.
        ch_names (list): List of channel names.
        montage (str): Montage name for the EEG data (e.g., 'biosemi128').
        source_sample_rate (int): Original sampling frequency of the EEG data.
        notch_freq (float): Notch filter frequency (default: 50 Hz).
        l_freq (float): Low cutoff frequency for bandpass filter (default: 0.5 Hz).
        h_freq (float): High cutoff frequency for bandpass filter (default: 40 Hz).
        verbose (bool): Verbose output.

    Returns:
        np.ndarray: Cleaned, normalized EEG data, shape (n_epochs, time_steps, channels).
    """
    # 1. Pick common channels
    keep = [ch for ch in ch_names if ch in raw.ch_names]
    raw.pick_channels(keep)
    if verbose:
        print(f"✔ Picked common channels ({len(keep)}): {keep}")

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage(montage))
    if verbose:
        print(f"✔ Montage set: {montage}.")
        
    # 3. Notch Filter at 50 Hz
    raw.notch_filter(freqs=notch_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Notch @ {notch_freq} Hz")

    # 4. Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    if verbose:
        print(f"✔ Bandpass filter applied ({l_freq}–{h_freq} Hz).")
    
    # 5. Set average reference for ICA
    raw.set_eeg_reference('average', projection=False)
    if verbose:
        print("✔ EEG re-referenced (average) before ICA.")
        
    # ICA . Performance drop when using ICA, might be due to already preprocessed data in their paper.
    raw_ica = raw.copy()
    ica = ICA(n_components=0.99, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [8]:
from mne.channels.interpolation import _make_interpolation_matrix

def interpolate_eeg_channels(raw, input_channels, input_montage, output_channels, output_montage):
    """
    Map EEG data from an original Raw object (with channels in 'input_montage')
    to a new set of channels ('output_channels' in 'output_montage') using
    spherical spline interpolation.

    This function builds a single interpolation matrix (via MNE's private
    function `_make_interpolation_matrix`), then multiplies it by the entire
    data matrix to achieve fast interpolation for long time series.

    Parameters
    ----------
    raw : mne.io.Raw
        The original EEG data object.
    input_channels : list
        A list of channel names in the raw data (e.g., the 128 Biosemi channel names).
    input_montage : str
        The montage name representing the layout of the original data
        (e.g., 'biosemi128').
    output_channels : list of str
        The target channel names (e.g., the standard 19 channels in the 10-20 system).
    output_montage : str
        The montage name for the target layout (e.g., 'standard_1020').

    Returns
    -------
    raw_out : mne.io.Raw
        The Raw object after spherical spline interpolation, containing
        len(output_channels) channels.
    """

    # 1) Create a copy of the raw object to avoid modifying the original data
    raw_copy = raw.copy()

    # 2) Verify that the number of channels matches the length of 'input_channels'
    if len(raw_copy.ch_names) != len(input_channels):
        raise ValueError(
            f"Number of channels in Raw ({len(raw_copy.ch_names)}) "
            f"does not match {len(input_channels)} in 'input_channels'."
        )

    # 3) Set the input montage (e.g., 'biosemi128') so that MNE knows the electrode locations
    montage_in = mne.channels.make_standard_montage(input_montage)
    raw_copy.set_montage(montage_in)

    # 4) Extract the (x, y, z) positions of each channel in the input montage
    pos_in = []
    for ch_info in raw_copy.info['chs']:
        xyz = ch_info['loc'][:3]
        pos_in.append(xyz)
    pos_in = np.array(pos_in)  # shape: (n_in, 3)
    n_in = len(pos_in)

    # 5) Retrieve the data from the raw object: (n_in x n_times), plus the time array
    data_in, times = raw_copy.get_data(return_times=True)
    n_times = data_in.shape[1]

    # 6) Prepare the output montage (e.g., 'standard_1020') and find the positions of the target channels
    montage_out = mne.channels.make_standard_montage(output_montage)
    pos_dict_out = montage_out.get_positions()['ch_pos']

    # Convert the target channel positions to a NumPy array of shape (n_out, 3)
    pos_out = np.array([pos_dict_out[ch] for ch in output_channels])
    n_out = len(pos_out)

    # 7) Build the spherical spline interpolation matrix W using MNE's internal function
    #    _make_interpolation_matrix(pos_from, pos_to) produces a matrix of shape (n_in, n_out)
    W = _make_interpolation_matrix(pos_in, pos_out)  # shape: (n_in, n_out)

    # 8) Apply W to all time points at once: data_out = W @ data_in
    #    data_in is (n_in x n_times), so the result is (n_out x n_times)
    print("W shape: ", W.shape)
    print("data_in shape: ", data_in.shape)
    data_out = W @ data_in  # shape: (n_out, n_times)

    # 9) Create a new Raw object with the interpolated data
    info_out = mne.create_info(
        ch_names=output_channels,
        sfreq=raw_copy.info['sfreq'],
        ch_types='eeg'
    )
    raw_out = mne.io.RawArray(data_out, info_out)

    # 10) Assign the output montage for correct electrode positions and visualization
    raw_out.set_montage(montage_out)

    return raw_out

In [9]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [11]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")
sub_id = 1
bad_subject_list = []
for disease_label in os.listdir(root):
    # Skip non-folder files
    if "BrainLat" in disease_label or "SYNAPSE" in disease_label:
        continue
    print(disease_label)
    disease_label_path = os.path.join(root, disease_label)
    for file_folder in os.listdir(disease_label_path):
        if file_folder == 'AR' or file_folder == 'CL':
            print(file_folder)
            file_folder_path = os.path.join(disease_label_path, file_folder)
            for file in os.listdir(file_folder_path):
                file_path = os.path.join(file_folder_path, file)
                set_file_path = ''
                if 'sub' in file_path or 'suj' in file_path:   # 4_MS has different folder name
                    set_file_path = os.path.join(file_path, 'eeg')
                    if not os.path.exists(set_file_path):  # same subjects in 4_MS has different folder structure
                        set_file_path = file_path
                    # print(set_file_path)
                    for set_file in os.listdir(set_file_path):
                        if set_file.endswith('.set'):
                            set_file_path = os.path.join(set_file_path, set_file)
                            print(set_file_path)
                            try:
                                # In their paper, preprocessing steps have been done including filtering(0.5-40Hz), re-referencing(Average), bad channel interpolation, and artifact rejection(ICA).
                                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)  
                                raw.pick_channels(common_channels)
                            except:
                                print('Load EEG data error\n')
                                bad_subject_list.append(set_file_path)
                                continue
                            raw.pick_channels(common_channels[:19])   # pick first 19 common channels, as we will interpolate to standard 19 channels later
                            data = raw.get_data().T
                            print("Cleaned data shape ", data.shape)
                            raw_sample_rate = int(raw.info['sfreq'])
                            T_raw = data.shape[0]
                            for fs in SAMPLE_RATE_LIST:
                                T_res = int(T_raw * fs / raw_sample_rate)
                                # compute number of segments
                                n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                                subject_segment_counts[sub_id][fs] += n_seg
                                N_total += n_seg
                                print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                            sub_id += 1
                            print('\n')
                        else:  # .ftd file will be read if .set file call it. no need to read it specifically
                            continue
    print("-------------------------------------\n")
    

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

1_AD
AR
BrainLat/dataset/EEG data/1_AD\AR\sub-30001\eeg\s6_sub-30001_rs-HEP_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Cleaned data shape  (159971, 19)
fs=200Hz: T_res=62488, STEP=200, n_seg=311
fs=100Hz: T_res=31244, STEP=200, n_seg=155
fs=50Hz: T_res=15622, STEP=200, n_seg=77


BrainLat/dataset/EEG data/1_AD\AR\sub-30002\eeg\s6_sub-30002_rs-HEP_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Cleaned data shape  (165534, 19)
fs=200Hz: T_res=64661, STEP=200, n_seg=322
fs=100Hz: T_res=32330, STEP=200, n_seg=160
fs=50Hz: T_res=16165, STEP=200, n_seg=79


BrainLat/dataset/EEG data/1_AD\AR\sub-30004\eeg\s6_sub-30004_rs-HEP_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

In [12]:
output_root = os.path.join('Processed', sub_folder_path, 'BrainLat')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\BrainLat\X.dat
y path: Processed\L400\BrainLat\y.dat


## PASS 2: Process and store segments into memmap

In [13]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

sub_id = 1
bad_subject_list = []
for disease_label in os.listdir(root):
    # Skip non-folder files
    if "BrainLat" in disease_label or "SYNAPSE" in disease_label:
        continue
    print(disease_label)
    disease_label_path = os.path.join(root, disease_label)
    for file_folder in os.listdir(disease_label_path):
        if file_folder == 'AR' or file_folder == 'CL':
            print(file_folder)
            file_folder_path = os.path.join(disease_label_path, file_folder)
            for file in os.listdir(file_folder_path):
                file_path = os.path.join(file_folder_path, file)
                set_file_path = ''
                if 'sub' in file_path or 'suj' in file_path:   # 4_MS has different folder name
                    set_file_path = os.path.join(file_path, 'eeg')
                    if not os.path.exists(set_file_path):  # same subjects in 4_MS has different folder structure
                        set_file_path = file_path
                    # print(set_file_path)
                    for set_file in os.listdir(set_file_path):
                        if set_file.endswith('.set'):
                            set_file_path = os.path.join(set_file_path, set_file)
                            print(set_file_path)
                            try:
                                # In their paper, preprocessing steps have been done including filtering(0.5-40Hz), re-referencing(Average), bad channel interpolation, and artifact rejection(ICA).
                                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)  
                                raw.pick_channels(common_channels)
                            except:
                                print('Load EEG data error\n')
                                bad_subject_list.append(set_file_path)
                                continue
                            raw.pick_channels(common_channels)   
                            raw = data_preprocessing(raw, common_channels, montage='biosemi128', notch_freq=50.0, l_freq=0.5, h_freq=45.0, verbose=True)
                            raw = interpolate_eeg_channels(raw, common_channels, 'biosemi128', standard_channels, 'standard_1020')
                            data = raw.get_data().T
                            print("Cleaned data shape ", data.shape)
                            raw_sample_rate = int(raw.info['sfreq'])
                            T_raw = data.shape[0]
                            total_seconds_all += data.shape[0] / raw_sample_rate
                            for fs in SAMPLE_RATE_LIST:
                                # resample to target fs
                                data_resampled = resample_time_series(data, raw_sample_rate, fs)
                                T_res, _ = data_resampled.shape
                                print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
                    
                                # overlapped sliding window with fixed STEP (in timestamps)
                                starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                                print(f"fs={fs}Hz: segments={len(starts)}")
                                
                                if 'HC' in disease_label:
                                    sub_label = 0
                                elif 'AD' in disease_label:
                                    sub_label = 1
                                elif 'FTD' in disease_label:
                                    sub_label = 2
                                elif 'PD' in disease_label:
                                    sub_label = 3
                                elif 'MS' in disease_label:
                                    sub_label = 4
                    
                                for s in starts:
                                    if cur >= N_total:
                                        raise RuntimeError("Exceeded predicted N_total.")
                    
                                    X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                                    y_mm[cur, 0] = float(sub_label)       # label
                                    y_mm[cur, 1] = float(sub_id)      # Global trial ID
                                    y_mm[cur, 2] = float(fs)          # sample rate (scale)
                                    cur += 1
                            sub_id += 1
                            print('\n')
                        else:  # .ftd file will be read if .set file call it. no need to read it specifically
                            continue
    print("-------------------------------------\n")
    

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

1_AD
AR
BrainLat/dataset/EEG data/1_AD\AR\sub-30001\eeg\s6_sub-30001_rs-HEP_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Picked common channels (128): ['A28', 'C20', 'D4', 'C22', 'D16', 'A21', 'A26', 'A30', 'A31', 'C6', 'C27', 'D1', 'C12', 'D15', 'C9', 'A5', 'D2', 'B15', 'B17', 'A7', 'D20', 'D24', 'C7', 'A2', 'B12', 'B30', 'B11', 'D11', 'B23', 'D19', 'A20', 'A1', 'B21', 'C2', 'A17', 'A13', 'C11', 'D31', 'C19', 'C21', 'C32', 'D13', 'B3', 'C25', 'D8', 'B25', 'A12', 'C31', 'B22', 'B27', 'C5', 'D10', 'C17', 'A14', 'C16', 'D21', 'D22', 'A32', 'C23', 'D28', 'D14', 'B16', 'B1', 'D23', 'B5', 'D3', 'B24', 'D12', 'D17', 'A18', 'C8', 'A27', 'A24', 'B6', 'B18', 'A29', 'A6', 'D6', 'C10', 'B9', 'C18', 'A11', 'A3', 'C14', 'D9', 'D30', 'D25', 'B19', 'A9', 'B20', 'B26', 'D5', 'B8', 'C3', 'D27', 'A16',

## Load and check the processed data

In [14]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 108206
  T = 400
  C = 19
  X_path = Processed\L400\BrainLat\X.dat
  y_path = Processed\L400\BrainLat\y.dat
-------------------------------------
Subject ID 001: X shape=(543, 400, 19), y shape=(543, 3)
Subject ID 002: X shape=(561, 400, 19), y shape=(561, 3)
Subject ID 003: X shape=(495, 400, 19), y shape=(495, 3)
Subject ID 004: X shape=(546, 400, 19), y shape=(546, 3)
Subject ID 005: X shape=(704, 400, 19), y shape=(704, 3)
Subject ID 006: X shape=(243, 400, 19), y shape=(243, 3)
Subject ID 007: X shape=(393, 400, 19), y shape=(393, 3)
Subject ID 008: X shape=(644, 400, 19), y shape=(644, 3)
Subject ID 009: X shape=(278, 400, 19), y shape=(278, 3)
Subject ID 010: X shape=(432, 400, 19), y shape=(432, 3)
Subject ID 011: X shape=(736, 400, 19), y shape=(736, 3)
Subject ID 012: X shape=(600, 400, 19), y shape=(600, 3)
Subject ID 013: X shape=(725, 400, 19), y shape=(725, 3)
Subject ID 014: X shape=(690, 400, 19), y shape=(690, 3)
Subject ID 015: X shape=(791, 400, 19), y sh